# FAIR CARE SURVEY Data Cleanup


In [1]:
from collections import Counter

import nltk
from nltk import ngrams
from nltk.util import everygrams
from nltk.corpus import stopwords

import os

import pandas as pd

import re
import statistics

# Get the root_path for this jupyter notebook repo.
repo_path = os.path.dirname(os.path.abspath(os.getcwd()))
# This is the path to a CSV file that provides configurations and metadata about the survey columns.
col_config_path = os.path.join(
    repo_path, 'files', 'IMLS-FAIR-CARE-Survey', 'imls-fair-care-survey-columns-config.csv',
)

raw_survey_path =  '/home/ekansa/oc-data/fair-care-survey.csv' # Keep this OUT of version control, has sensitive info
processed_survey_path = '/home/ekansa/oc-data/fair-care-survey-processed.csv' # Keep this OUT of version control, has sensitive info
col_summary_path = '/home/ekansa/oc-data/fair-care-survey-col-summary.csv'
token_freq_path = '/home/ekansa/oc-data/fair-care-token-freq.csv'
ngram_freq_path = '/home/ekansa/oc-data/fair-care-ngram-freq.csv'
semantic_clusters_path = '/home/ekansa/oc-data/fair-care-semantic-clusters.csv'

# Skip one row. 
df = pd.read_csv(raw_survey_path, skiprows=1)

# Delete a few un-wanted rows in the header of the source dataset
bad_rows = [0]
df.drop(index=bad_rows, inplace=True)


## Step 1: Clean up the raw survey dataset by dropping unwanted columns and renaming columns for easier use.

This step loads an external CSV data file ``imls-fair-care-survey-columns-config.csv`` that provides configuration and metadata about each column. Edit that datafile to change how columns are named or dropped.

In [2]:

# The df_config dataframe contains column configuration and metadata
df_config = pd.read_csv(col_config_path)

col_index = 0
drop_cols = []
rename_cols = {}
for col in df.columns.tolist():
    config_index = (
        (df_config['raw_column'] == col)
        |
        (df_config['raw_column'].str.startswith(col[0:50]) & df_config['raw_column'].str.endswith(col[-50:]))
    )
    if len(df_config[config_index].index) != 1:
        # We didn't fina a unique match. The exported data probably had some sort of schema change.
        raise ValueError(f"[{col_index}] found {len(df_config[config_index].index)} MISSING: '{col}'")

    section = df_config[config_index]['Section'].iloc[0]
    rename = df_config[config_index]['Working_Column_Name'].iloc[0]
    # print(f"[{col_index}] {col[0:45]} --- config section {section}")
    col_index += 1
    if section == 'SKIP':
        drop_cols.append(col)
    if not isinstance(rename, str) or rename == '' or rename == 'nan':
        continue
    rename_cols[col] = rename

# Drop the columns that our configuration classified as "SKIP"
df.drop(columns=drop_cols, inplace=True)

# Rename the columns based on information in the config.
df.rename(columns=rename_cols, inplace=True)

# Save the processed survey data into a new file
df.to_csv(processed_survey_path, index=False)

## Step 2: Define some functions to look up configuration metadata about each column

The external CSV data file ``imls-fair-care-survey-columns-config.csv`` provides metadata about each column. These functions make retrieval of that metadata more convenient.

In [3]:
column_metadata_keys = [
    'orig_column_index',
    'Question Number',
    'Important for Analysis',
    'Section',
    'Sub-Section',
]

def get_column_metadata(column_name, df_config=df_config, column_metadata_keys=column_metadata_keys):
    """Get a dict of column metadata for a given column_name"""
    config_index = (df_config['Working_Column_Name'] == column_name)
    if len(df_config[config_index].index) != 1:
        # We didn't find a matching column name
        return None
    row = df_config[config_index].iloc[0]
    col_metadata = {meta_col: row[meta_col] for meta_col in df_config.columns.tolist() if meta_col in column_metadata_keys}
    return col_metadata

def get_column_names_for_section(section, sub_section=None, df_config=df_config):
    """Gets a list of column names for a given section (and optional sub_section)"""
    config_index = (df_config['Section'] == section)
    if sub_section:
         config_index &= (df_config['Sub-Section'] == sub_section)
    column_names = df_config[config_index]['Working_Column_Name'].tolist()
    return column_names


# Make a list of columns to use as (composite) keys to include for join opperations.
key_cols = get_column_names_for_section(section='KEY')

## Make a datatable providing basic statistics about each column

The following makes a `fair-care-survey-col-summary.csv` data table that provides a simple summary of the contents of each column in the FAIR+CARE response dataset.

In [4]:
# Download stopwords and punctation tables if not already available
nltk.download('stopwords')
nltk.download('punkt_tab')

col_summary_rows = []
# Iterate over all the columns in the response dataframe so we can make a summary of each col.
for col in df.columns.tolist():
    # We're mainly interesting in summarizing responses that are not left blank.
    not_null_index = ~df[col].isnull() # Set up an index for non-blank values of the col
    char_lengths = [] # this will be a list of character lengths for this col
    word_counts = [] # this will be a list of the number of words in each column.
    # Iterate over each non-blank value for this col.
    for col_str in df[not_null_index][col]:
        col_str = str(col_str)
        char_lengths.append(len(col_str))
        words = col_str.split()
        word_counts.append(len(words))

    col_metadata = get_column_metadata(column_name=col)
    if not col_metadata:
        # We don't have metadata for this column, but it would be good
        # to record that.
        col_metadata = {}
    # Make a dict with the summary information for this specific col.
    row = {'column': col}
    # Add the column metadata to the row
    for key, val in col_metadata.items():
        row[key] = val
    row_data = {
        'count_not_blank': len(df[not_null_index].index),
        'count_unique': len(df[not_null_index][col].unique().tolist()),
        'max_word_count': max(word_counts),
        'mean_word_count': statistics.mean(word_counts),
        'max_char_length': max(char_lengths),
        'mean_char_length': statistics.mean(char_lengths),
    }
    # Add the data about the column to the row
    for key, val in row_data.items():
        row[key] = val
    col_summary_rows.append(row)
# Turn the col_summary_rows into a dataframe called df_col_sum
df_col_sum = pd.DataFrame(col_summary_rows)
# Save the column summary dataframe to a CSV file.
df_col_sum.to_csv(col_summary_path, index=False)
# Show some examples of the column summary dataframe 
df_col_sum.head(20)



[nltk_data] Downloading package stopwords to /home/ekansa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ekansa/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,column,orig_column_index,Question Number,Important for Analysis,Section,Sub-Section,count_not_blank,count_unique,max_word_count,mean_word_count,max_char_length,mean_char_length
0,Finished,6,NaN,NaN,Metadata,General,787,2,1,1.000000,5,4.538755
1,Recorded Date,7,NaN,NaN,Metadata,General,787,787,2,2.000000,19,19.000000
2,Response ID,8,NaN,NaN,KEY,KEY,787,787,1,1.000000,17,17.000000
3,Demo: Age Range,18,-9.0,NaN,Demographics,General,681,7,1,1.000000,5,4.735683
4,Demo: Gender,19,-8.0,NaN,Demographics,General,673,3,4,1.044577,25,5.313522
5,Demo: Race/Ethnicity,20,-7.0,NaN,Demographics,General,668,15,15,3.227545,111,19.757485
6,"Demo: Race/Ethnicity, Other, Text",21,-7.0,NaN,Demographics,General,29,25,11,1.724138,54,11.344828
7,Edu: Highest Degree,22,-6.0,NaN,Demographics,General,682,6,5,1.585044,47,10.636364
8,"Edu: Other qualifications, Text",23,-6.0,NaN,Demographics,General,34,27,10,2.882353,59,20.411765
9,"Edu: Highest Degree Discipline, Text",24,-5.0,NaN,Demographics,General,574,216,12,1.627178,95,16.031359


## Make a datatable that summarizes the frequency of different words for each text column.

The following makes a `fair-care-token-freq.csv` data table that provides a frequency summary of each token (similar to word) for each column in the FAIR+CARE response dataset. This frequency is summarized for each survey unique survey response, text column, and word.

In [5]:
# Make a list of columns. These are columns with lots of variety of responses and where there's variability in how
# many words are used in the responses.
cols_with_free_text = (
    ((df_col_sum['count_unique'] / df_col_sum['count_not_blank']) >= 0.33) 
    & (df_col_sum['max_word_count'] > (df_col_sum['mean_word_count'] * 1.5) )
)
text_cols = df_col_sum[cols_with_free_text]['column'].unique().tolist()

List of free text columns is:

In [6]:
for col in text_cols:
    print(col)

Demo: Race/Ethnicity, Other, Text
Edu: Other qualifications, Text
Edu: Highest Degree Discipline, Text
Work: Work setting, Other, Text
Region Focus: Work/research, Africa, Text
Region Focus: Work/research, Asia, Text
Region Focus: Work/research, Europe, Text
Region Focus: Work/research, N America, Text
Region Focus: Work/research, S America, Text
Region Focus: Work/research, Other, Text
Response Type: Org, Text
Data Role: How You Work with Data, Text
Data Org: Other, Text
Data Store: Data Storage, Other, Text
Findable: Yes, Text
Findable: No, Explain, Text
Findable: Sometimes, Explain, Text
Identifiers: Other, Text
Metadata Search: Metadata Search Method, Text
Supplemental Data: Other, Text
Cost Paid: Other, Text
Accessible: Additional info, Text
Why Acquire: Other, Text
How Acquire: Other, Text
Understanding: Problems Understanding Data
Understanding: Problems Understanding Data, Text
Shared Terms: Shared Terms, Co-Created, Text
Shared Terms: Shared Terms, Other, Text
Media Used: Prop

In [7]:

def get_filtered_token_counts(text):
    """Get words and their frequency counts from a text, excluding "stop" words"""
    text = str(text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = text.replace('  ', ' ')
    # Convert text to lowercase and tokenize
    tokens = nltk.word_tokenize(text.lower())
    # Remove stop words
    stop_words = set(stopwords.words('english'))
    filtered_tokens = [token for token in tokens if token not in stop_words]
    # Remove tokens that are numbers from this.
    no_number_tokens = []
    for token in filtered_tokens:
        if token == '0':
            continue
        try:
            num_str = int(token)
        except:
            num_str = None
        if num_str:
            continue
        if len(token) < 2:
            # stip really short tokens, since these are probably noise.
            continue
        no_number_tokens.append(token)
    token_counts = Counter(no_number_tokens)
    return token_counts


token_freq_rows = []
# Iterate over all the columns in the response dataframe so we can make a summary of each col.
for col in df.columns.tolist():
    if col not in text_cols:
        continue
    # We're mainly interesting in summarizing responses that are not left blank.
    not_null_index = ~df[col].isnull() # Set up an index for non-blank values of the col
    for _, row in df[not_null_index].iterrows():
        # get all the unique tokens and their counts.
        token_counts = get_filtered_token_counts(row[col])
        for token, count in token_counts.items():
            # Copy all the key columns needed for good joins
            new_row = {col_key:row.get(col_key) for col_key in key_cols}
            col_metadata = get_column_metadata(column_name=col)
            if not col_metadata:
                # We don't have metadata for this column, but it would be good
                # to record that.
                col_metadata = {}
            # Add the column metadata to the new_row
            for key, val in col_metadata.items():
                new_row[key] = val
            # Now add the column information
            new_row['column'] = col
            new_row['token'] = token
            new_row['token_count'] = count
            token_freq_rows.append(new_row)
# Make a dataframe of all of the token_freq_rows
df_token_freq_raw = pd.DataFrame(token_freq_rows)

# Now aggregate the token count, grouping by responder_cols values, the column, and the token.
token_freq_group_cols = key_cols + ['column', 'token']
df_token_freq = df_token_freq_raw.groupby(token_freq_group_cols, as_index=False).sum()

# Save the results and display some examples
df_token_freq.to_csv(token_freq_path, index=False)
df_token_freq.head(20)


,Response ID,column,token,orig_column_index,Question Number,Important for Analysis,Section,Sub-Section,token_count
0,R_10Cnr999hIh7JiM,Application of Ethical Frameworks: Text,allow,152,40.0,0,CARE,Ethics,1
1,R_10Cnr999hIh7JiM,Application of Ethical Frameworks: Text,analysis,152,40.0,0,CARE,Ethics,1
2,R_10Cnr999hIh7JiM,Application of Ethical Frameworks: Text,framework,152,40.0,0,CARE,Ethics,1
3,R_10Cnr999hIh7JiM,Application of Ethical Frameworks: Text,try,152,40.0,0,CARE,Ethics,1
4,R_10Cnr999hIh7JiM,Application of Ethical Frameworks: Text,understand,152,40.0,0,CARE,Ethics,1
5,R_10Cnr999hIh7JiM,"CARE: Define Indigenous Data, Text",ethnographic,100,21.0,Yes,CARE,General,1
6,R_10Cnr999hIh7JiM,"CARE: Define Indigenous Data, Text",group,100,21.0,Yes,CARE,General,1
7,R_10Cnr999hIh7JiM,"CARE: Define Indigenous Data, Text",mythologies,100,21.0,Yes,CARE,General,1
8,R_10Cnr999hIh7JiM,"CARE: Define Indigenous Data, Text",publications,100,21.0,Yes,CARE,General,1
9,R_10Cnr999hIh7JiM,Collaboration Processes: Collaboration Process...,really,145,34.0,0,CARE,Authority to Control,1


## Make a datatable that summarizes the frequency of different n-grams for each text column.

The following makes a `fair-care-ngram-freq.csv` data table that provides a frequency summary of each ngram (phrases) for each column in the FAIR+CARE response dataset. This frequency is summarized for each survey unique survey response, text column, and ngram.

In [8]:
def get_everygrams(text, max_len=5):
    """Gets a list of everygrams (2-4) for a text value"""
    stop_words = set(stopwords.words('english'))
    text = str(text)
    text = re.sub(r'[^\w\s]', ' ', text)
    text = text.replace('  ', ' ')
    text = text.lower()
    text_split = text.split()
    every_grams = {} # will be keyed by the ngram length
    for act_gram_raw in list(everygrams(text_split, max_len=max_len)):
        act_gram = [token for token in act_gram_raw if len(token) > 0]
        ngram_len = len(act_gram)
        if ngram_len < 2:
            # we're skipping single tokens (or less).
            continue
        non_stop = [token for token in act_gram if token not in stop_words]
        if not non_stop:
            # All the tokens in this act_gram are stop words, so skip
            continue
        # make a string of the ngram and add it to the every_grams 
        ngram_str = ' '.join(act_gram)
        if not every_grams.get(ngram_len):
            every_grams[ngram_len] = []
        every_grams[ngram_len].append(ngram_str)
    return every_grams



ngram_freq_rows = []
# Iterate over all the columns in the response dataframe so we can make a summary of each col.
for col in df.columns.tolist():
    if col not in text_cols:
        continue
    # We're mainly interesting in summarizing responses that are not left blank.
    not_null_index = ~df[col].isnull() # Set up an index for non-blank values of the col
    for _, row in df[not_null_index].iterrows():
        # get all the ngrams in this text (we're getting 2, 3, 4 and 5 grams ngrams)
        every_grams = get_everygrams(row[col])
        for ngram_len, ngrams in every_grams.items():
            for ngram in ngrams:
                # Copy all the key columns needed for good joins
                new_row = {col_key:row.get(col_key) for col_key in key_cols}
                col_metadata = get_column_metadata(column_name=col)
                if not col_metadata:
                    # We don't have metadata for this column, but it would be good
                    # to record that.
                    col_metadata = {}
                # Add the column metadata to the new_row
                for key, val in col_metadata.items():
                    new_row[key] = val
                # Now add the column information
                new_row['column'] = col
                new_row['ngram_len'] = ngram_len
                new_row['ngram'] = ngram
                new_row['ngram_count'] = 1
                ngram_freq_rows.append(new_row)

# Make a dataframe of all of the ngram_freq_rows
df_ngram_freq_raw = pd.DataFrame(ngram_freq_rows)

# Now aggregate the ngram count, grouping by responder_cols values, the column, and the ngram_len and the ngram
ngram_freq_group_cols = key_cols +  ['column', 'ngram_len', 'ngram',]
df_ngram_freq = df_ngram_freq_raw.groupby(ngram_freq_group_cols, as_index=False).sum()

# Save the results and display some examples
df_ngram_freq.to_csv(ngram_freq_path, index=False)
df_ngram_freq.tail(20)

,Response ID,column,ngram_len,ngram,orig_column_index,Question Number,Important for Analysis,Section,Sub-Section,ngram_count
396791,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,the information was accurate or,77,14.0,0,FAIR,Interoperable,1
396792,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,to access questionable data quality,77,14.0,0,FAIR,Interoperable,1
396793,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,tools to access questionable data,77,14.0,0,FAIR,Interoperable,1
396794,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,trustworthy varied types of data,77,14.0,0,FAIR,Interoperable,1
396795,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,types of data encountered data,77,14.0,0,FAIR,Interoperable,1
396796,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,unclear if the information was,77,14.0,0,FAIR,Interoperable,1
396797,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,varied types of data encountered,77,14.0,0,FAIR,Interoperable,1
396798,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,was accurate or trustworthy varied,77,14.0,0,FAIR,Interoperable,1
396799,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,was locked or scrambled needing,77,14.0,0,FAIR,Interoperable,1
396800,R_8sZlHxYMBWQrCHo,Understanding: Problems Understanding Data,5,was missing or why encrypted,77,14.0,0,FAIR,Interoperable,1


## Make a datatable that semantically clusters responses for each text column.

The following makes a `fair-care-semantic-clusters.csv` data table that provides a frequency summary of each ngram (phrases) for each column in the FAIR+CARE response dataset. This frequency is summarized by each unique demographic of respondents.

In [9]:
from bertopic import BERTopic

from bertopic.representation import KeyBERTInspired
# from bertopic.representation import PartOfSpeech
from bertopic.representation import MaximalMarginalRelevance

from sentence_transformers import SentenceTransformer, util

import numpy as np

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.cluster import AgglomerativeClustering

sentence_model = SentenceTransformer('distilbert-base-nli-stsb-quora-ranking')

MAX_NUM_CLUSTRERS = 10

def choose_classifier(X, max_num_clusters=MAX_NUM_CLUSTRERS):
    X1 = X / (X**2).sum(axis=-1, keepdims=True)
    vv = []
    cc = np.arange(2, len(X))
    for nclusters in cc:
        if nclusters >  max_num_clusters:
            continue
        km_model = KMeans(
            n_clusters=nclusters,
            max_iter=100,
            n_init=10,
        ).fit(X1)
        labels = km_model.labels_
        v = silhouette_score(X1, labels)
        vv.append(v)
    finish_clusters = cc[np.argmax(vv)]
    return KMeans(
        n_clusters=finish_clusters,
        max_iter=100,
        n_init=10,
    ).fit(X1)


def get_agglom_custer_id(resp, df_cluster):
    """Gets bert classifiation for a specific response, as a dict"""
    rep_index = df_cluster['response'] == resp
    if len(df_cluster[rep_index].index) != 1:
        return None
    row = df_cluster[rep_index].iloc[0]
    return row['cluster_id']


def get_bert_classification_for_response(resp, df_bert, col_suffix):
    """Gets bert classifiation for a specific response, as a dict"""
    rep_index = df_bert['Document'] == resp
    if len(df_bert[rep_index].index) != 1:
        return None
    row = df_bert[rep_index].iloc[0]
    return {
        f'bert_name_{col_suffix}': row['Name'],
        f'bert_prob_{col_suffix}': row['Probability'],
        f'bert_is_repr_{col_suffix}': row['Representative_document'],
        f'bert_top_words_{col_suffix}': row['Top_n_words'],
    }


In [10]:
# Create objects for using BERT
vectorizer_model = CountVectorizer(stop_words="english")
representation_model_key = KeyBERTInspired()
# representation_model_pos = PartOfSpeech("en_core_web_sm")
representation_model_mmr = MaximalMarginalRelevance(diversity=0.3)

semantic_cluster_rows = []
# Iterate over all the columns in the response dataframe so we can make a summary of each col.
for col in df.columns.tolist():
    if col not in text_cols:
        continue
    # We're mainly interesting in summarizing responses that are not left blank.
    not_null_index = ~df[col].isnull() # Set up an index for non-blank values of the col
    col_responses = df[not_null_index][col].unique().tolist()

    # Do the sentence transformer classifications
    print(f'Make sentence transformer embeddings for {len(col_responses)} responses to {col}')
    embeddings = sentence_model.encode(col_responses, show_progress_bar=True, convert_to_numpy=True)
    classifier = choose_classifier(embeddings)

    # Make a non-numpy embeddings
    embeddings_bert = sentence_model.encode(col_responses, show_progress_bar=False)
    clustering_model = AgglomerativeClustering(
        n_clusters=None, distance_threshold=1.5
    ) 
    clustering_model.fit(embeddings_bert)
    cluster_assignment = clustering_model.labels_

    # Make a dataframe of the hierarchic clusters.
    cluster_rows = []
    for resp_id, cluster_id in enumerate(cluster_assignment):
        cluster_row = {
            'cluster_id':  cluster_id,
            'response': col_responses[resp_id],
        }
        cluster_rows.append(cluster_row)
    df_cluster = pd.DataFrame(cluster_rows)


    # Use different methods of BERTopic to organize responses
    topic_model_plain = BERTopic(embedding_model=sentence_model)
    topics_plain, _ = topic_model_plain.fit_transform(col_responses, embeddings_bert)
    topic_model_plain.update_topics(col_responses, vectorizer_model=vectorizer_model)
    df_bert_plain = topic_model_plain.get_document_info(col_responses)
    
    topic_model_key = BERTopic(embedding_model=sentence_model, representation_model=representation_model_key)
    topics_key, _ = topic_model_key.fit_transform(col_responses, embeddings_bert)
    topic_model_key.update_topics(col_responses, vectorizer_model=vectorizer_model)
    df_bert_key = topic_model_key.get_document_info(col_responses)

    if False:
        # skip this, too many, and this doesn't actually install well.
        topic_model_pos = BERTopic(embedding_model=sentence_model, representation_model=representation_model_pos)
        topics_pos, _ = topic_model_pos.fit_transform(col_response, embeddings_bert)
        topic_model_pos.update_topics(col_responses, vectorizer_model=vectorizer_model)
        df_bert_pos = topic_model_pos.get_document_info(col_responses)

    topic_model_mmr = BERTopic(embedding_model=sentence_model, representation_model=representation_model_mmr)
    topics_mmr, _ = topic_model_mmr.fit_transform(col_responses, embeddings_bert)
    topic_model_mmr.update_topics(col_responses, vectorizer_model=vectorizer_model)
    df_bert_mmr = topic_model_mmr.get_document_info(col_responses)

    df_berts = {
        'plain': df_bert_plain,
        'key': df_bert_key,
        # 'pos': df_bert_pos,
        'mmr': df_bert_mmr,
    }
    
    for i, (v, resp) in enumerate(zip(embeddings, col_responses)):
        semantic_group_np = classifier.predict(v[np.newaxis])
        semantic_group_list = semantic_group_np.tolist()
        semantic_group = semantic_group_list[0]
        resp_index = (df[col] == resp)
        for _, row in df[resp_index].iterrows():
            # Copy all the key columns needed for good joins
            new_row = {col_key:row.get(col_key) for col_key in key_cols}
            col_metadata = get_column_metadata(column_name=col)
            if not col_metadata:
                # We don't have metadata for this column, but it would be good
                # to record that.
                col_metadata = {}
            # Add the column metadata to the new_row
            for key, val in col_metadata.items():
                new_row[key] = val
            # Now add the column information
            new_row['column'] = col
            new_row['response'] = resp
            
            # Add word counts to this, because word counts may indicate something about semantic interest.
            resp_lower = re.sub(r'[^\w\s]', ' ', resp)
            resp_lower = resp_lower.lower()
            new_row['word_count'] = len(resp_lower.split())

            # Add the semantic group ID from the sentence transformer.
            new_row['semantic_group_st'] = semantic_group

            cluster_id = get_agglom_custer_id(resp, df_cluster)
            if cluster_id:
                new_row['semantic_agglom_cluster_id'] = cluster_id

            # Now add BERT classifications
            for col_suffix, df_bert in df_berts.items():
                bert_dict = get_bert_classification_for_response(resp, df_bert, col_suffix)
                if not bert_dict:
                    continue
                for b_k, b_v in bert_dict.items():
                    new_row[b_k] = b_v
            
            semantic_cluster_rows.append(new_row)
# Make a dataframe of all of the token_freq_rows
df_semantic_raw = pd.DataFrame(semantic_cluster_rows)

# Now aggregate the token count, grouping by responder_cols values, the column, and the token.
semantic_group_cols = key_cols + ['column']
df_semantic = df_semantic_raw.groupby(semantic_group_cols, as_index=False).first()
df_semantic.sort_values(by=['orig_column_index', 'semantic_group_st'], inplace=True)

# Save the results and display some examples
df_semantic.to_csv(semantic_clusters_path, index=False)
print(f'Saved {len(df_semantic.index)} row dataset to {semantic_clusters_path}')

df_semantic.tail(20)

Make sentence transformer embeddings for 25 responses to Demo: Race/Ethnicity, Other, Text


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Make sentence transformer embeddings for 27 responses to Edu: Other qualifications, Text


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Make sentence transformer embeddings for 216 responses to Edu: Highest Degree Discipline, Text


...

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Make sentence transformer embeddings for 96 responses to Survey Comments: Text


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Saved 7242 row dataset to /home/ekansa/oc-data/fair-care-semantic-clusters.csv


,Response ID,column,orig_column_index,Question Number,Important for Analysis,Section,Sub-Section,response,word_count,semantic_group_st,...,bert_is_repr_plain,bert_top_words_plain,bert_name_key,bert_prob_key,bert_is_repr_key,bert_top_words_key,bert_name_mmr,bert_prob_mmr,bert_is_repr_mmr,bert_top_words_mmr
2640,R_3HSebRrFLUiDrwJ,Survey Comments: Text,165,48.0,None,General,General,There are quite a few questions,6,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.893451,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...
2783,R_3OPjhc1NVMXP1wn,Survey Comments: Text,165,48.0,None,General,General,Some of the questions are hard to answer when ...,38,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...
3067,R_3d4FRMShzUQnl5L,Survey Comments: Text,165,48.0,None,General,General,I feel vaguely concerned that there are not ve...,20,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...
3136,R_3fdPEIFWvM8ptOU,Survey Comments: Text,165,48.0,None,General,General,"It wasn't too long. As mentioned above, it was...",28,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...
3339,R_3s0aNkhS59lJZnB,Survey Comments: Text,165,48.0,None,General,General,"This took two of us 1 hour, and we struggled t...",26,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...
3854,R_541SJEjTpDNTIfN,Survey Comments: Text,165,48.0,None,General,General,This is long... but it's important information.,8,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.806893,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.868061,False,survey - data - questions - work - indigenous ...
4239,R_5GlwLYhM6UZ2IFv,Survey Comments: Text,165,48.0,None,General,General,I'm afraid it's a bit too long,9,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.688855,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.876997,False,survey - data - questions - work - indigenous ...
4430,R_5QGux45sP3rDcbe,Survey Comments: Text,165,48.0,None,General,General,A bit long,3,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.726771,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.765126,False,survey - data - questions - work - indigenous ...
5003,R_5yf1vnwtbreKU9r,Survey Comments: Text,165,48.0,None,General,General,The length was a bit long. I could not have do...,33,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.947688,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indigenous ...
5625,R_72SkrsNtEINFeCA,Survey Comments: Text,165,48.0,None,General,General,A couple of the questions seem specific to som...,14,3,...,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,0.920098,False,survey - data - questions - work - indigenous ...,0_survey_data_questions_work,1.000000,False,survey - data - questions - work - indige